<a href="https://colab.research.google.com/github/timfitz04/Business-Analytics-Dissertation/blob/main/Wicklow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://officialwicklowgaa.ie/fixtures-results/

In this file Wicklow league tables were scraped and collected then stored.

Team names were then cleaned so the dataset would  merge with gaapitchfinder dataset adding longitudal and latiudal data to the orignial data that was scraped. Teams that did not match after cleaning were mannually matched.

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
os.chdir("/content/drive/MyDrive/Dissertation/Dub")
!pwd

/content/drive/MyDrive/Dissertation/Dub


In [15]:
!pip install bs4

In [16]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import re
time.sleep(5)



compIDs_by_year = {
    2025: {
        "Division 1": 230741,
        "Division 2": 230743,
        "Division 3": 230745,
        "Division 4": 230747,
        "Division 5A": 236521,
        "Division 5B": 236523,
        "Division 6A": 235267,
        "Division 6B": 235269,
    },
    2024: {
        "Division 1": 209223,
        "Division 2": 209225,
        "Division 3": 209227,
        "Division 4": 209283,
        "Division 5A": 213569,
        "Division 5B": 213571,
        "Division 6A": 213573,
        "Division 6B": 213577,
    },
    2022: {
        "Division 1": 168945,
        "Division 2": 168947,
        "Division 3": 169157,
        "Division 4": 169159,
        "Division 5": 175647,
        "Division 6S": 175653,
        "Division 6E": 175655,
        "Division 6W": 175663,
    },
    2019: {
        "Division 1": 114279,
        "Division 2": 114281,
        "Division 3": 114503,
        "Division 4": 114821,
        "Division 5SW": 114287,
        "Division 5NE": 114289,
        "Division 6SW": 114515,
        "Division 6NE": 114291,
    },
    2018: {
        "Division 1": 95085,
        "Division 2": 95087,
        "Division 3": 95089,
        "Division 4": 95091,
        "Division 5SW1": 95513,
        "Division 5NE1": 95515,
        "Division 5SW2": 95507,
        "Division 5NE2": 95509,
    },

}

all_rows = []

for year, divisions in compIDs_by_year.items():

    for division_name, compid in divisions.items():

        url = (
            f"https://officialwicklowgaa.ie/fixtures-results/fixtures-results-ajax/?countyBoardID=32&ccAC=1&compID={compid}&leagueTable=Y&resultsOnly=Y&reverseDateOrder=Y&orderTBCLast=Y"
        )

        response = requests.get(url)
        time.sleep(5)
        soup = BeautifulSoup(response.text, "html.parser")


        tables = pd.read_html(str(soup))
        if len(tables) == 0:
            print(f"No table for {division_name} {year}")
            continue

        df = tables[0]

        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        rename_map = {
            "p": "pld",
            "team": "team",
            "w": "w",
            "d": "d",
            "l": "l",
            "f": "pf",
            "a": "pa",
            "pts": "pts"
        }
        df = df.rename(columns=rename_map)


        # Calculate points difference
        df["pd"] = df["pf"].astype(int) - df["pa"].astype(int)

        # Add division + year
        df["Division"] = division_name
        df["Year"] = year
        df["County"] = "Wicklow"

        df = df[["County", "Division", "Year", "team", "pld", "w", "d", "l", "pf", "pa", "pd", "pts"]]

        all_rows.append(df)


# ---- FINAL OUTPUT ----
Wicklow = pd.concat(all_rows, ignore_index=True)

print(Wicklow)


/tmp/ipykernel_2496/1396100355.py:79: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_2496/1396100355.py:79: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_2496/1396100355.py:79: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_2496/1396100355.py:79: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_2496/1396100355.p

      County       Division  Year  \
0    Wicklow     Division 1  2025   
1    Wicklow     Division 1  2025   
2    Wicklow     Division 1  2025   
3    Wicklow     Division 1  2025   
4    Wicklow     Division 1  2025   
..       ...            ...   ...   
333  Wicklow  Division 5NE2  2018   
334  Wicklow  Division 5NE2  2018   
335  Wicklow  Division 5NE2  2018   
336  Wicklow  Division 5NE2  2018   
337  Wicklow  Division 5NE2  2018   

                                                  team  pld  w  d   l   pf  \
0                                              Rathnew    7  5  0   2  116   
1                                   Éire Óg Greystones    7  5  0   2  103   
2                                          Baltinglass    7  5  0   2  113   
3                                          Blessington    7  3  1   3  114   
4                                             Tinahely    7  3  1   3  105   
..                                                 ...  ... .. ..  ..  ...   
333      

/tmp/ipykernel_2496/1396100355.py:79: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))


In [17]:
def clean_club_name(name):
    name = name.lower()


    # remove trailing numbers (Thomas Davis 2 → Thomas Davis)
    name = re.sub(r"\s*\d+$", "", name)

    # remove reserve/second team indicators e.g. "2nd", "B", "C"
    name = re.sub(r"\b(2nd|second|team|b|c)\b", "", name, flags=re.IGNORECASE)

    # strip GAA-specific suffixes and organisation labels
    name = re.sub(r"\b(gaa|clg|club|lgfa|ladies)\b", "", name, flags=re.IGNORECASE)

    # remove punctuation + multiple spaces
    name = re.sub(r"[^\w\s]", "", name)
    name = " ".join(name.split())

    return name.strip()

# after scraping into Dub dataframe:
Wicklow["clean_team"] = Wicklow["team"].apply(clean_club_name)

In [18]:
coords = pd.read_csv("gaapitchfinder_data.csv")
coords["clean_team"] = coords["Club"].apply(clean_club_name)
coords = coords[coords["County"] == "Wicklow"].copy()
coords = coords.drop_duplicates(
    subset="clean_team",
    keep="first"
)
coords = coords[["clean_team", "Latitude", "Longitude"]]



In [19]:
coords

,clean_team,Latitude,Longitude
261,wicklow,52.852503,-6.337295
404,arklow geraldines ballymoney,52.820962,-6.178137
406,annacurra,52.841607,-6.353948
407,an tochar,53.059263,-6.229983
408,aughrim,52.852554,-6.335164
409,arklow rock parnells,52.790124,-6.176280
410,ashford,53.012328,-6.109534
411,avoca,52.853743,-6.215919
412,avondale,52.923067,-6.241915
413,ballinacor,52.946562,-6.333260


In [20]:
manual_map = {
    "arklow geraldinesballymoneyna ngearaltaighbaile mhuine": "arklow geraldines ballymoney",
    "naomh teagainkiltegan": "naomh teagain kiltegan",
    "emmet carn an bhua":"carnew emmets",
    "abhainn dala":"avondale",
    "shillelagh coolboy":"coolboy",
    "donardglen":"donard glen",
    "ath na fuinseoige":"ashford",
    "cnoc an eanaigh":"knockananna",
    "stratford grange con ath na sraide grainseach choinn":"stratford grangecon",
    "cill bhride":"kilbride",
    "st patricks kilcoole":"st patricks wicklow",
    "lackenkilbride":"lacken",
    "éire óg greystones":"eire og greystones",
    "an tóchar":"an tochar",
    "naomh teagain kiltegan":"naomh teagáin kiltegan",
    "abhainn dála":"avondale",
    "stratford grange con áth na sráide grainseach choinn":"stratford grangecon"

}


Wicklow["clean_team"] = Wicklow["clean_team"].replace(manual_map)

In [21]:
Wicklow = Wicklow.merge(coords, on=["clean_team"], how="left")

In [24]:
Wicklow.to_csv('/content/drive/MyDrive/Dissertation/Wicklow/wicklow_leagues.csv', index=False)